In [1]:
import requests
import pandas as pd
from io import StringIO
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
import numpy as np

In [2]:
# List of all Teams and IDs
team_ids = [108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 158]
team_names = [
    "Yankees", "Red Sox", "Blue Jays", "Rays", "Orioles",
    "White Sox", "Guardians", "Tigers", "Royals", "Twins",
    "Astros", "Mariners", "Angels", "Rangers", "Athletics",
    "Braves", "Marlins", "Mets", "Phillies", "Nationals",
    "Cubs", "Reds", "Brewers", "Pirates", "Cardinals",
    "D-backs", "Rockies", "Dodgers", "Padres", "Giants"
]

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

season = 2024

all_standard = []
all_statcast1 = []
all_statcast2 = []

In [3]:
# Importing the data from all teams
for team_id in team_ids:
    url = f"https://baseballsavant.mlb.com/team/{team_id}?view=statcast&nav=hitting&season={season}"
    print(f"Fetching data for team {team_id} ...")
    response = requests.get(url, headers=headers)
    try:
        tables = pd.read_html(StringIO(response.text))
        standard, statcast1, statcast2 = tables[0], tables[1], tables[2]
    except Exception as e:
        print(f"Failed to get data for team {team_id}: {e}")
        continue

    # Clean column names
    unwanted = ['Unnamed: 1_level_0', 'Unnamed: 0_level_0', 'Statcast', 'Standard Stats','Unnamed: 26_level_1']

    def clean_column(col):
        for u in unwanted:
            col = col.replace(u, '').strip()
        col = ' '.join(col.split())
        return col

    for df in [standard, statcast1, statcast2]:
        new_columns = {}
        for col in df.columns:
            col_str = ' '.join(col).strip() if isinstance(col, tuple) else str(col)
            new_columns[col] = clean_column(col_str)
        df.rename(columns=new_columns, inplace=True)

    # Add team info for later
    standard['TeamID'] = team_id
    statcast1['TeamID'] = team_id
    statcast2['TeamID'] = team_id

    all_standard.append(standard)
    all_statcast1.append(statcast1)
    all_statcast2.append(statcast2)

Fetching data for team 108 ...
Fetching data for team 109 ...
Fetching data for team 110 ...
Fetching data for team 111 ...
Fetching data for team 112 ...
Fetching data for team 113 ...
Fetching data for team 114 ...
Fetching data for team 115 ...
Fetching data for team 116 ...
Fetching data for team 117 ...
Fetching data for team 118 ...
Fetching data for team 119 ...
Fetching data for team 120 ...
Fetching data for team 121 ...
Fetching data for team 133 ...
Fetching data for team 134 ...
Fetching data for team 135 ...
Fetching data for team 136 ...
Fetching data for team 137 ...
Fetching data for team 138 ...
Fetching data for team 139 ...
Failed to get data for team 139: No tables found
Fetching data for team 140 ...
Fetching data for team 141 ...
Fetching data for team 142 ...
Fetching data for team 143 ...
Fetching data for team 144 ...
Fetching data for team 145 ...
Fetching data for team 146 ...
Fetching data for team 147 ...
Fetching data for team 158 ...


In [4]:
# Organizing the data
def flatten_columns(df):
    df.columns = [
        ' '.join(col).strip() if isinstance(col, tuple) else str(col).strip()
        for col in df.columns
    ]
    return df

unwanted = ['Unnamed: 1_level_0', 'Unnamed: 0_level_0', 'Statcast', 'Standard Stats', 'Unnamed: 26_level_1']

def clean_column(col):
    for u in unwanted:
        col = col.replace(u, '').strip()
    col = ' '.join(col.split())
    return col

def clean_columns(df):
    df.columns = [clean_column(col) for col in df.columns]
    return df

def fix_player_column(df):
    for col in df.columns:
        if 'Player' in col:
            df = df.rename(columns={col: 'Player'})
            break
    return df

def final_cleanup(df):
    df = flatten_columns(df)
    df = clean_columns(df)
    df = fix_player_column(df)
    df = df.loc[:, ~df.columns.duplicated()]
    return df

standard = final_cleanup(standard)
statcast3 = final_cleanup(statcast3)


NameError: name 'statcast3' is not defined

In [ ]:
# Merge statcast data
statcast = pd.merge(statcast1, statcast2, on=['Player','TeamID'], how='inner')
statcast = pd.merge(statcast, statcast3, on=['Player', 'TeamID'], how='inner')

In [ ]:
# Calculate "Expected Runs" for standard csv file
for col in ['2B','3B','HR','H','BB','AB']:
    if col not in standard.columns:
        standard[col] = 0

standard['TB'] = (
    standard['2B'] * 2 +
    standard['3B'] * 3 +
    standard['HR'] * 4 +
    (standard['H'] - standard['2B'] - standard['3B'] - standard['HR'])
)

standard['Runs'] = ((standard['H'] + standard['BB']) * standard['TB']) / (standard['AB'] + standard['BB'])
standard['Runs'] = standard['Runs'].round()

In [ ]:
# Export to CSV
standard.to_csv('Standard_Batting_All_Teams.csv', index=False)
statcast.to_csv('Statcast_All_Teams.csv', index=False)

In [ ]:
# Feature lists and betas
hit_features = ['Whiff %','Batted Balls','Hard Hit %','Exit Velocity','XBA','Zone Contact %','Chase %', 'Zone Swing %', 'Straight %', 'FB %', 'LD %', 'Barrel %']
walk_features = ['Chase %', 'Batted Balls', 'Pitches', 'Whiff %', 'Zone Swing %', 'Zone Contact %']
bats_features = ['Pitches', 'Batted Balls']
bases_features = ['XSLG','Whiff %','Chase %','Barrels', 'Hard Hit %', 'Exit Velocity', 'Straight %', 'Batted Balls', 'Barrel %']

hit_betas = {'Whiff %': -2,'Batted Balls': 5,'Hard Hit %': 5,'Exit Velocity': 3,'XBA': 4,'Zone Contact %': .5,'Chase %': -3, 'Zone Swing %': .4, 'Straight %': 3, 'FB %': 1, 'LD %': 2,'Barrel %':2}
walk_betas = {'Chase %': -4, 'Batted Balls': 5, 'Pitches': 6, 'Whiff %': -2, 'Zone Swing %': 1, 'Zone Contact %': 2}
bats_betas = {'Pitches': 1, 'Batted Balls': 1,}
bases_betas = {'XSLG': 1,'Whiff %': -2,'Chase %': -4,'Barrels': 5, 'Hard Hit %': 3, 'Exit Velocity': 2, 'Straight %': 2, 'Batted Balls': 1, 'Barrel %': 5}

In [ ]:
# Calculating inputs to data science experiments
all_features = set(hit_features + walk_features + bats_features + bases_features)
missing = [f for f in all_features if f not in statcast.columns]
if missing:
    print(f"Missing columns in statcast DataFrame: {missing}")
else:
    scaler = StandardScaler()
    statcast_scaled = statcast.copy()
    statcast_scaled[list(all_features)] = scaler.fit_transform(statcast[list(all_features)])
    
    def compute_statcast_score(df, features, betas):
        return sum(df[feat] * betas[feat] for feat in features)
    statcast['Hit_Score']   = compute_statcast_score(statcast_scaled, hit_features, hit_betas)
    statcast['Walk_Score']  = compute_statcast_score(statcast_scaled, walk_features, walk_betas)
    statcast['Bats_Score']  = compute_statcast_score(statcast_scaled, bats_features, bats_betas)
    statcast['Bases_Score'] = compute_statcast_score(statcast_scaled, bases_features, bases_betas)
    statcast['Hit_Score']   = statcast['Hit_Score'].round(3)
    statcast['Walk_Score']  = statcast['Walk_Score'].round(3)
    statcast['Bats_Score']  = statcast['Bats_Score'].round(3)
    statcast['Bases_Score'] = statcast['Bases_Score'].round(3)

Missing columns in statcast DataFrame: ['Pitches', 'Barrel %']


In [ ]:
rows_to_remove = team_names + ["MLB"]

# Filter out all rows where 'Player' is in rows_to_remove
statcast = statcast[~statcast['Player'].isin(rows_to_remove)].reset_index(drop=True)

In [ ]:
# Ensure 'Pitches' is numeric (sometimes it's read as string)
statcast['Pitches'] = pd.to_numeric(statcast['Pitches'], errors='coerce')

# Remove anyone with less than 100 pitches
statcast = statcast[statcast['AB'] >= 100].reset_index(drop=True)

KeyError: 'Pitches'

In [ ]:
# Export to CSV
standard.to_csv('Standard_Batting_All_Teams.csv', index=False)
statcast.to_csv('Statcast_All_Teams.csv', index=False)

In [ ]:
# Select relevant columns
statcast_scores = statcast[['Player', 'TeamID', 'Hit_Score', 'Walk_Score', 'Bats_Score', 'Bases_Score']]
standard_stats = standard[['Player', 'TeamID', 'H', 'BB', 'AB', 'TB']]

# Merge on Player and TeamID
merged = pd.merge(statcast_scores, standard_stats, on=['Player', 'TeamID'], how='inner')

# --- 2. Plot relationships between scores and real stats ---
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

sns.regplot(x='Hit_Score', y='H', data=merged, ax=axs[0, 0], line_kws={'color': 'red'})
axs[0, 0].set_title('Hit Score vs Hits')

sns.regplot(x='Walk_Score', y='BB', data=merged, ax=axs[0, 1], line_kws={'color': 'red'})
axs[0, 1].set_title('Walk Score vs Walks')

sns.regplot(x='Bases_Score', y='TB', data=merged, ax=axs[1, 1], line_kws={'color': 'red'})
axs[1, 1].set_title('Bases Score vs Total Bases')

sns.regplot(x='Bats_Score', y='AB', data=merged, ax=axs[1, 0], line_kws={'color': 'red'})
axs[1, 0].set_title('Bats Score vs At Bats')

plt.tight_layout()
plt.show()

# --- 3. Fit and print regression equations for each stat ---
def fit_equation(x, y, label):
    x = np.array(x).reshape(-1, 1)
    y = np.array(y)
    reg = LinearRegression().fit(x, y)
    print(f"{label}: y = {reg.coef_[0]:.3f} * x + {reg.intercept_:.3f}")
    return reg.coef_[0], reg.intercept_

hit_coef, hit_int = fit_equation(merged['Hit_Score'], merged['H'], "Hits")
walk_coef, walk_int = fit_equation(merged['Walk_Score'], merged['BB'], "Walks")

bases_coef, bases_int = fit_equation(merged['Bases_Score'], merged['TB'], "Total Bases")
bats_coef, bats_int = fit_equation(merged['Bats_Score'], merged['AB'], "At Bats")

# --- 4. (Optional) Print summary table of equations ---
print("\nSummary of Equations:")
print(f"Hits        : H  = {hit_coef:.3f} * Hit_Score   + {hit_int:.3f}")
print(f"Walks       : BB = {walk_coef:.3f} * Walk_Score  + {walk_int:.3f}")
print(f"Total Bases : TB = {bases_coef:.3f} * Bases_Score + {bases_int:.3f}")
print(f"At Bats     : AB = {bats_coef:.3f} * Bats_Score  + {bats_int:.3f}")


KeyError: "['Hit_Score', 'Walk_Score', 'Bats_Score', 'Bases_Score'] not in index"